# MiniGPT from scratch: Colab runner

This notebook reproduces the whole project from a clean checkout:
tests → architecture evidence → training → generation → evaluation → attention maps → ablations → API.

**Runtime → Change runtime type → T4 GPU** before running. The full pipeline takes about 30–40 minutes on a T4.

## 1. Clone and install

In [ ]:
# >>> set this to your repository <<<
REPO_URL = "https://github.com/<your-user>/transformer-gpt-assignment.git"

import os
if not os.path.exists("transformer-gpt-assignment"):
    !git clone -q $REPO_URL transformer-gpt-assignment
%cd transformer-gpt-assignment
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Unit tests and the tiny overfit gate

In [ ]:
!python -m pytest -q

In [ ]:
!python -m src.train --overfit

## 3. Architecture evidence (Figures 1–5)

In [ ]:
!python -m experiments.attention_numeric_example
!python -m experiments.shape_trace
!python -m experiments.positional_encoding_viz

In [ ]:
from IPython.display import Image, display
for f in ["fig2_causal_mask_3tok.png", "fig2_causal_mask_12.png", "fig4_positional_encoding.png"]:
    display(Image(f"outputs/figures/{f}"))

### Lesson 01: RNN vs self-attention timing (on the GPU) and Lesson 06: MLM vs CLM

In [ ]:
!python -m experiments.why_transformers
!python -m experiments.mask_comparison --steps 1500
for f in ["why_transformers.png", "fig_mlm_vs_clm_masks.png", "fig_mlm_vs_clm_losses.png"]:
    display(Image(f"outputs/figures/{f}"))

## 4. Train the baseline MiniGPT (Figures 5, 6, 8)

Configuration: `configs/tiny_gpt.yaml`. Any value can be overridden with `--set section.key=value`.

*Skip this cell* if you only want to run inference with a published checkpoint (see section 9).

In [ ]:
!python -m src.train

In [ ]:
import json
display(Image("outputs/figures/tiny_gpt_loss.png"))
print(json.dumps(json.load(open("outputs/logs/tiny_gpt_summary.json")), indent=2))

## 5. Generation from fixed prompts (CLI)

In [ ]:
!python -m src.generate --prompt "ROMEO:" --max-new-tokens 300 --seed 0
!python -m src.generate --prompt "JULIET:" --max-new-tokens 300 --seed 0
!python -m src.generate --prompt "To be, or not to be" --max-new-tokens 300 --temperature 0 

## 6. Evaluation: full validation perplexity, 5 fixed prompts, decoding comparison

In [ ]:
!python -m experiments.evaluate
from IPython.display import Markdown
display(Markdown(open("outputs/samples/evaluation.md").read()))

## 7. Attention maps from the trained model (Figure 7)

In [ ]:
!python -m experiments.attention_visualization
import glob
for f in sorted(glob.glob("outputs/figures/fig7_attention_layer*.png")):
    display(Image(f))

## 8. Ablations (Figure 9)

Depth (2/4/6 layers), heads (2/4/8), context length (64/128/256), each 2000 steps with everything else fixed.
Add `position` to the list for the optional sinusoidal vs learned vs none study.

In [ ]:
!python -m experiments.ablation --studies depth heads context --steps 2000
display(Markdown(open("outputs/ablation/ablation_results.md").read()))
for s in ["depth", "heads", "context"]:
    display(Image(f"outputs/ablation/ablation_{s}.png"))

In [ ]:
# optional
!python -m experiments.ablation --studies position --steps 2000

## 9. Inference-only path: load a published checkpoint

If the checkpoint is not in the repo, set `CHECKPOINT_URL` to a GitHub release asset or a public Google Drive
share link; `src.utils.load_checkpoint` downloads it on first use.

In [ ]:
# os.environ["CHECKPOINT_URL"] = "https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing"
!python -m src.generate --checkpoint checkpoints/tiny_gpt.pt --prompt "ROMEO:" --seed 1

## 10. REST API inside Colab

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "api.app:app", "--port", "8000"])
time.sleep(8)
print(requests.get("http://127.0.0.1:8000/health").json())
r = requests.post("http://127.0.0.1:8000/generate",
                  json={"prompt": "ROMEO:", "max_new_tokens": 200, "temperature": 0.8, "seed": 0})
print(r.json()["generated_text"])
server.terminate()

## 11. Save the results

In [ ]:
!zip -qr results.zip outputs checkpoints/tiny_gpt.pt
from google.colab import files
files.download("results.zip")